# Calibration-scene RGB quicklooks

Presentation-ready RGB views of the three scenes used for the cross-sensor SNR diagnostics: **EnMAP** (Agadez, Niger), **PRISMA** (Northern State, Sudan), and **Tanager-1** (Sudan).

The EnMAP and PRISMA RGBs are the existing full-column products generated from their Level-1 calibration scenes. The Tanager RGB is made directly from the Basic TOA-radiance product used for the SNR reference. Its red outline marks the reference-SNR ROI: all 607 columns and the first 234 rows (`0:607,0:234`).


In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from osgeo import gdal

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'scripts').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUTPUT_DIR = REPO_ROOT / 'notebooks' / 'outputs' / 'calibration_scene_rgb_quicklooks'
REFERENCE_SNR_ROI_LABEL = 'Reference-SNR ROI: all columns, top 234 rows'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENES = {
    'EnMAP': {
        'rgb_path': REPO_ROOT / 'test_data' / 'enmap' / 'Agadez_Niger_20220712' / 'L1B-DT0000001584_20220712T104302Z_001_V010502_20251017T093724Z' / 'full-column' / 'L1B_DT0000001584_001_20220712T104302Z_20220712T104307Z_full-column_RGB.tif',
        'caption': 'Agadez, Niger | 12 Jul 2022',
    },
    'PRISMA': {
        'rgb_path': REPO_ROOT / 'test_data' / 'prisma' / 'Northern_State_Sudan_20200401' / '20200401085313_20200401085318' / 'full-column' / 'PRS_L1_STD_OFFL_20200401085313_20200401085318_0001_full-column_rgb.tif',
        'caption': 'Northern State, Sudan | 1 Apr 2020',
    },
    'Tanager-1': {
        'radiance_path': REPO_ROOT / 'test_data' / 'tanager' / '20250509_090323_87_4001' / 'basic_radiance_hdf5__20250509_090323_87_4001_basic_radiance_hdf5.h5',
        'caption': 'Sudan | 9 May 2025',
        'snr_roi': (0, 607, 0, 234),  # x0, x1, y0, y1: all columns, homogeneous top rows
    },
}

for sensor, config in SCENES.items():
    for key, value in config.items():
        if key.endswith('_path') and not value.exists():
            raise FileNotFoundError(f'{sensor}: missing {key}: {value}')

print(f'Writing PNGs to: {OUTPUT_DIR}')


In [ ]:
def read_rgb_geotiff(path: Path) -> np.ndarray:
    """Return a three-band GeoTIFF as a float RGB image in [0, 1]."""
    dataset = gdal.Open(str(path), gdal.GA_ReadOnly)
    if dataset is None:
        raise RuntimeError(f'Could not read {path}')
    data = dataset.ReadAsArray().astype(np.float32)
    if data.shape[0] != 3:
        raise ValueError(f'Expected three bands in {path.name}, found {data.shape[0]}')
    rgb = np.moveaxis(data, 0, -1)
    if np.nanmax(rgb) > 1:
        rgb /= 255.0
    # PRISMA's valid swath is surrounded by zero-valued no-data pixels.
    rgb[np.all(rgb <= 0, axis=-1)] = np.nan
    return percentile_rgb(rgb)


def percentile_rgb(rgb: np.ndarray, low: float = 2, high: float = 98) -> np.ndarray:
    scaled = np.empty_like(rgb, dtype=np.float32)
    for channel in range(3):
        band = rgb[..., channel]
        valid = band[np.isfinite(band)]
        lo, hi = np.percentile(valid, [low, high])
        scaled[..., channel] = np.clip((band - lo) / (hi - lo), 0, 1) if hi > lo else 0
    return np.nan_to_num(scaled, nan=0.0)


def read_tanager_rgb(path: Path) -> np.ndarray:
    """Read only visual bands from the Basic radiance cube used for the SNR reference."""
    radiance_dataset = 'HDFEOS/SWATHS/HYP/Data Fields/toa_radiance'
    nodata_dataset = 'HDFEOS/SWATHS/HYP/Data Fields/nodata_pixels'
    with h5py.File(path, 'r') as handle:
        radiance = handle[radiance_dataset]
        wavelengths = np.asarray(radiance.attrs['wavelengths'], dtype=float)
        indices = [int(np.argmin(np.abs(wavelengths - target))) for target in (665, 565, 490)]
        rgb = np.stack([radiance[index] for index in indices], axis=-1).astype(np.float32)
        nodata = np.asarray(handle[nodata_dataset]) != 0
    rgb[nodata] = np.nan
    return percentile_rgb(rgb)


rgb_images = {
    'EnMAP': read_rgb_geotiff(SCENES['EnMAP']['rgb_path']),
    'PRISMA': read_rgb_geotiff(SCENES['PRISMA']['rgb_path']),
    'Tanager-1': read_tanager_rgb(SCENES['Tanager-1']['radiance_path']),
}

for sensor, image in rgb_images.items():
    print(f'{sensor}: {image.shape[1]} columns × {image.shape[0]} rows')


In [ ]:
# Individual, high-resolution PNGs. The Tanager frame includes the exact reference-SNR footprint.
for sensor, image in rgb_images.items():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image)
    ax.set_title(f"{sensor} calibration scene (visual quicklook)\n{SCENES[sensor]['caption']} | independent display stretch; not a common radiance scale", fontsize=13)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=3))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10,
                bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 4})
    ax.axis('off')
    fig.savefig(OUTPUT_DIR / f"calibration_rgb_{sensor.lower().replace('-', '_')}.png", dpi=300, bbox_inches='tight', pad_inches=0.03)
    plt.show()


In [ ]:
# One landscape panel for slides and reports.
fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (sensor, image) in zip(axes, rgb_images.items()):
    ax.imshow(image)
    ax.set_title(f"{sensor}\n{SCENES[sensor]['caption']}", fontsize=14)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=2.5))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 3})
    ax.axis('off')

fig.text(0.5, 0.01, 'Visual quicklooks: independent rendered-RGB stretches; not radiometrically comparable.', ha='center', fontsize=11)
panel_path = OUTPUT_DIR / 'calibration_rgb_panel.png'
fig.savefig(panel_path, dpi=300, bbox_inches='tight', pad_inches=0.03)
print(f'Saved combined panel: {panel_path}')
plt.show()


## Common-wavelength, common-radiance RGB

The previous figures are independently contrast-stretched for visual clarity. The cells below retain those figures and add a comparable alternative: each sensor is sampled at the nearest bands to **650 / 560 / 490 nm**, converted to **µW cm$^{-2}$ sr$^{-1}$ nm$^{-1}$**, and displayed with the same fixed radiance limits for every channel and sensor.

The fixed 8–20 µW cm$^{-2}$ sr$^{-1}$ nm$^{-1}$ range is deliberately not adjusted per scene. Pixels outside it are clipped, so differences in brightness and colour reflect the selected common radiance scale rather than per-image statistics.


In [ ]:
COMMON_RGB_WAVELENGTHS_NM = (650.0, 560.0, 490.0)
COMMON_RADIANCE_LIMITS = (8.0, 20.0)  # µW cm^-2 sr^-1 nm^-1, applied to every channel and sensor


def nearest_index(wavelengths: np.ndarray, target_nm: float) -> int:
    return int(np.argmin(np.abs(np.asarray(wavelengths, dtype=float) - target_nm)))


def read_enmap_radiance_rgb(scene_dir: Path):
    """Read only the EnMAP VNIR RGB bands and convert to common radiance units."""
    from scripts.satellites import enmap_utils

    vnir_path, _, metadata_path = enmap_utils.find_enmap_files(str(scene_dir))
    vnir_meta, _ = enmap_utils.parse_metadata_vnir_swir(metadata_path)
    wavelengths = np.array([band['cw_nm'] for band in vnir_meta], dtype=float)
    dataset = gdal.Open(vnir_path, gdal.GA_ReadOnly)
    bands, actual_nm = [], []
    for target_nm in COMMON_RGB_WAVELENGTHS_NM:
        index = nearest_index(wavelengths, target_nm)
        meta = vnir_meta[index]
        dn = dataset.GetRasterBand(index + 1).ReadAsArray().astype(np.float32)
        # W m^-2 sr^-1 nm^-1 -> µW cm^-2 sr^-1 nm^-1
        bands.append((meta['gain'] * dn + meta['offset']) * 100.0)
        actual_nm.append(meta['cw_nm'])
    return np.stack(bands, axis=-1), np.array(actual_nm)


def read_prisma_radiance_rgb(path: Path):
    """Read only PRISMA VNIR RGB bands, matching prisma_read's wavelength handling."""
    cube_path = 'HDFEOS/SWATHS/PRS_L1_HCO/Data Fields/VNIR_Cube'
    cw_path = 'KDP_AUX/Cw_Vnir_Matrix'
    with h5py.File(path, 'r') as handle:
        cube = handle[cube_path]
        cw_matrix = np.asarray(handle[cw_path], dtype=float)
        n_bands = cube.shape[1]  # PRISMA L1 VNIR cube is BIL: rows, bands, columns

        active = np.where(np.nanmax(cw_matrix, axis=0) > 0)[0]
        first, last = int(active[0]), int(active[-1])
        start = max(0, first - max(0, n_bands - (last - first + 1)))
        cw_columns = np.arange(start, start + n_bands)
        cw_slice = cw_matrix[:, cw_columns]
        valid = np.nanmax(cw_slice, axis=0) > 0
        cube_indices = np.arange(n_bands)[valid]
        wavelengths = np.nanmean(cw_slice[:, valid], axis=0)
        if wavelengths[0] > wavelengths[-1]:
            cube_indices = cube_indices[::-1]
            wavelengths = wavelengths[::-1]

        bands, actual_nm = [], []
        scale = float(handle.attrs['ScaleFactor_Vnir'])
        offset = float(handle.attrs['Offset_Vnir'])
        for target_nm in COMMON_RGB_WAVELENGTHS_NM:
            index = nearest_index(wavelengths, target_nm)
            dn = cube[:, int(cube_indices[index]), :].astype(np.float32)
            # W m^-2 sr^-1 µm^-1 -> µW cm^-2 sr^-1 nm^-1
            radiance = (dn / scale - offset) * 0.1
            bands.append(np.rot90(radiance, k=-1))  # same spatial orientation as prisma_read
            actual_nm.append(wavelengths[index])
    return np.stack(bands, axis=-1), np.array(actual_nm)


def read_tanager_radiance_rgb(path: Path):
    """Read only Tanager's nearest RGB bands and convert to common radiance units."""
    radiance_dataset = 'HDFEOS/SWATHS/HYP/Data Fields/toa_radiance'
    nodata_dataset = 'HDFEOS/SWATHS/HYP/Data Fields/nodata_pixels'
    with h5py.File(path, 'r') as handle:
        cube = handle[radiance_dataset]
        wavelengths = np.asarray(cube.attrs['wavelengths'], dtype=float)
        nodata = np.asarray(handle[nodata_dataset]) != 0
        indices = [nearest_index(wavelengths, target_nm) for target_nm in COMMON_RGB_WAVELENGTHS_NM]
        # W m^-2 sr^-1 µm^-1 -> µW cm^-2 sr^-1 nm^-1
        rgb = np.stack([cube[index] * 0.1 for index in indices], axis=-1).astype(np.float32)
    rgb[nodata] = np.nan
    return rgb, wavelengths[indices]


def fixed_radiance_rgb(radiance_rgb: np.ndarray) -> np.ndarray:
    vmin, vmax = COMMON_RADIANCE_LIMITS
    scaled = np.clip((radiance_rgb - vmin) / (vmax - vmin), 0, 1)
    return np.nan_to_num(scaled, nan=0.0)


enmap_scene_dir = SCENES['EnMAP']['rgb_path'].parents[1]
prisma_l1_path = REPO_ROOT / 'test_data' / 'prisma' / 'Northern_State_Sudan_20200401' / '20200401085313_20200401085318' / 'PRS_L1_STD_OFFL_20200401085313_20200401085318_0001.he5'

common_radiance, selected_wavelengths = {}, {}
common_radiance['EnMAP'], selected_wavelengths['EnMAP'] = read_enmap_radiance_rgb(enmap_scene_dir)
common_radiance['PRISMA'], selected_wavelengths['PRISMA'] = read_prisma_radiance_rgb(prisma_l1_path)
common_radiance['Tanager-1'], selected_wavelengths['Tanager-1'] = read_tanager_radiance_rgb(SCENES['Tanager-1']['radiance_path'])

for sensor in common_radiance:
    actual = ', '.join(f'{value:.1f}' for value in selected_wavelengths[sensor])
    print(f'{sensor}: nearest bands = {actual} nm; shape = {common_radiance[sensor].shape[:2]}')

common_rgb_images = {sensor: fixed_radiance_rgb(image) for sensor, image in common_radiance.items()}

for sensor, image in common_rgb_images.items():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image)
    actual = ' / '.join(f'{value:.1f}' for value in selected_wavelengths[sensor])
    ax.set_title(f"{sensor} | common radiance scale\nnearest bands: {actual} nm | fixed R/G/B: {COMMON_RADIANCE_LIMITS[0]:g}–{COMMON_RADIANCE_LIMITS[1]:g} µW cm$^{{-2}}$ sr$^{{-1}}$ nm$^{{-1}}$", fontsize=13)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=3))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 3})
    ax.axis('off')
    fig.savefig(OUTPUT_DIR / f"calibration_rgb_common_radiance_{sensor.lower().replace('-', '_')}.png", dpi=300, bbox_inches='tight', pad_inches=0.03)
    plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (sensor, image) in zip(axes, common_rgb_images.items()):
    ax.imshow(image)
    actual = ' / '.join(f'{value:.1f}' for value in selected_wavelengths[sensor])
    ax.set_title(f"{sensor}\n{actual} nm", fontsize=14)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=2.5))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 3})
    ax.axis('off')

fig.text(0.5, 0.01, f'Fixed radiance mapping for every R, G, and B channel: {COMMON_RADIANCE_LIMITS[0]:g}–{COMMON_RADIANCE_LIMITS[1]:g} µW cm$^{{-2}}$ sr$^{{-1}}$ nm$^{{-1}}$ (no per-scene adjustment).', ha='center', fontsize=11)
panel_path = OUTPUT_DIR / 'calibration_rgb_common_radiance_panel.png'
fig.savefig(panel_path, dpi=300, bbox_inches='tight', pad_inches=0.03)
print(f'Saved common-radiance panel: {panel_path}')
plt.show()




## Interpretable dynamic-radiance RGB

This is the radiance-based counterpart to the visual quicklooks. For each sensor and each channel independently, the display interval is calculated from valid pixels as the **2nd to 98th percentile** of radiance, [P2(L), P98(L)]. The displayed channel is clip((L − P2) / (P98 − P2), 0, 1).

The annotation on every image reports its exact R/G/B radiance intervals in µW cm$^{-2}$ sr$^{-1}$ nm$^{-1}$. This improves within-scene visibility, but remains a dynamic, per-scene stretch; use the fixed-scale panel above for cross-sensor radiance comparison.


In [ ]:
CHANNEL_NAMES = ('R', 'G', 'B')
RADIANCE_UNIT = 'µW cm$^{-2}$ sr$^{-1}$ nm$^{-1}$'


def dynamic_radiance_rgb(radiance_rgb: np.ndarray, percentiles=(2.0, 98.0)):
    """Independently map each radiance channel from P2–P98 to display values 0–1."""
    scaled = np.empty_like(radiance_rgb, dtype=np.float32)
    limits = []
    for channel in range(3):
        band = radiance_rgb[..., channel]
        valid = band[np.isfinite(band)]
        low, high = np.percentile(valid, percentiles)
        scaled[..., channel] = np.clip((band - low) / (high - low), 0, 1) if high > low else 0
        limits.append((float(low), float(high)))
    return np.nan_to_num(scaled, nan=0.0), np.asarray(limits)


def dynamic_interval_text(sensor: str, limits: np.ndarray) -> str:
    wavelengths = selected_wavelengths[sensor]
    rows = ['Dynamic radiance stretch: P2–P98']
    for channel, wavelength, (low, high) in zip(CHANNEL_NAMES, wavelengths, limits):
        rows.append(f'{channel} ({wavelength:.1f} nm): {low:.2f}–{high:.2f}')
    rows.append(RADIANCE_UNIT)
    return '\n'.join(rows)


dynamic_rgb_images, dynamic_limits = {}, {}
for sensor, radiance in common_radiance.items():
    dynamic_rgb_images[sensor], dynamic_limits[sensor] = dynamic_radiance_rgb(radiance)

for sensor, image in dynamic_rgb_images.items():
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(image)
    ax.set_title(f'{sensor} | dynamic radiance scale (per-channel P2–P98)', fontsize=14)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=3))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 3})
    ax.text(0.02, 0.02, dynamic_interval_text(sensor, dynamic_limits[sensor]), transform=ax.transAxes,
            va='bottom', color='white', fontsize=9, bbox={'facecolor': 'black', 'alpha': 0.72, 'pad': 4})
    ax.axis('off')
    fig.savefig(OUTPUT_DIR / f"calibration_rgb_dynamic_radiance_{sensor.lower().replace('-', '_')}.png", dpi=300, bbox_inches='tight', pad_inches=0.03)
    plt.show()

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (sensor, image) in zip(axes, dynamic_rgb_images.items()):
    ax.imshow(image)
    ax.set_title(f'{sensor} | dynamic P2–P98', fontsize=14)
    if sensor == 'Tanager-1':
        x0, x1, y0, y1 = SCENES[sensor]['snr_roi']
        ax.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='red', linewidth=2.5))
        ax.text(12, 224, REFERENCE_SNR_ROI_LABEL, color='white', fontsize=10, bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 3})
    ax.text(0.02, 0.02, dynamic_interval_text(sensor, dynamic_limits[sensor]), transform=ax.transAxes,
            va='bottom', color='white', fontsize=7.5, bbox={'facecolor': 'black', 'alpha': 0.72, 'pad': 3})
    ax.axis('off')

panel_path = OUTPUT_DIR / 'calibration_rgb_dynamic_radiance_panel.png'
fig.savefig(panel_path, dpi=300, bbox_inches='tight', pad_inches=0.03)
print(f'Saved dynamic-radiance panel: {panel_path}')
plt.show()
